En este notebook vamos a hacer distintas **queries a nuestra base de datos ElasticSearch**.

Como ya tenemos creado el índice "Shakespeare" con todas sus obras, aprovecharemos para hacer consultas en él.

Recordemos que estamos en otro notebook, por lo que tiene una IP diferente, así que, permitamos que este notebook acceda a nuestro server Elasticsearch:

In [1]:
!curl ipecho.net/plain

34.106.214.197

Instalemos también la librería elasticsearch:

In [2]:
!pip install elasticsearch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 524.6/524.6 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 5.1 MB/s eta 0:00:00


# Ahora, vamos a repasar las **posibles consultas** que se pueden hacer a una **base de datos ElasticSearch**

## Un pequeño Setup primero que nos facilitará el trabajo:

Declaramos un **client** y unos **métodos auxiliares para nuestras consultas**:

In [3]:
server_es_ip = '35.238.251.163' # La IP del servidor ElasticSearch

In [4]:
from elasticsearch import Elasticsearch
from dateutil.parser import parse as parse_date

# Creamos el cliente y conectamos al server
es = Elasticsearch("http://{}:9200".format(server_es_ip))

es.info()

ObjectApiResponse({'name': 'elastic-1', 'cluster_name': 'elasticsearch', 'cluster_uuid': 'fNJXS-1UQZeWSzq61ewiOQ', 'version': {'number': '8.14.1', 'build_flavor': 'default', 'build_type': 'deb', 'build_hash': '93a57a1a76f556d8aee6a90d1a95b06187501310', 'build_date': '2024-06-10T23:35:17.114581191Z', 'build_snapshot': False, 'lucene_version': '9.10.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'})

In [5]:
def print_hits(results):
  """
    Función para printear los resultados de una consulta

    Args:
    results (dict): El objeto de resultados de la búsqueda devuelto por Elasticsearch.

    Returns:
    None
  """
  print(results['hits'])
  # Llama a la función print_search_stats para printear estadísticas de los resultados
  print_search_stats(results)

  # Iteramos osbre cada resultado (hit)
  for hit in results['hits']['hits']:
    # Aseguramos que si no existe el campo _type (A partir de la versión 7. de Elasticsearch type no está presente en hit), utiliza _doc
    # Esto es porque en las nuevas versiones de ES, todos los tipos son doc
    doc_type = hit.get('_type', '_doc')
    # Extraemos los campos de cada consulta y los formateamos
    print('/%s/%s/%s [%s]: |%s| %s - %s' % (
            hit['_index'],
            doc_type,
            hit['_id'],
            hit['_score'],
            hit['_source']['play_name'].split('\n')[0],
            hit['_source']['speaker'].split('\n')[0],
            hit['_source']['text_entry']))

  # Damos estilo al print para mejorar legibilidad
  print('=' * 80)

def print_search_stats(results):
  """
    Imprime estadísticas básicas de los resultados de una búsqueda.

    Args:
    results (dict): Objeto de resultados de la búsqueda devuelto por Elasticsearch.

    Returns:
    None
  """
  print('=' * 80)
  print('Total %d found in %dms' % (results['hits']['total']['value'], results['took']))
  print('-' * 80)


In [6]:
def search_query(query):
  """
    Ejecuta una llamada a Elasticsearch con una query específica.

    Args:
        query (str): String con la query para Elasticsearch

    Returns:
        None
  """
  # Definimos que el index será siempre "shakespeare". En caso de trabajar con más de un índice o base de datos, podemos modificar la función para que index sea un parámetro
  print_hits(es.search(index='shakespeare', q=query))

In [7]:
def search_query_body(body):
  """
    Ejecuta una llamada a Elasticsearch con un cuerpo de query especificado

    Args:
        body (dict): Cuerpo de la consulta que se utilizará en la request

    Returns:
        None
  """
  print_hits(es.search(index='shakespeare', body=body))

## **Consultas**

### Buscar en el campo `_all`

ElasticSearch permite hacer una búsqueda en todos los campos a la vez, gracias a un campo especial llamado `_all` disponible en las queries:

In [8]:
search_query("*")

{'total': {'value': 10000, 'relation': 'gte'}, 'max_score': 1.0, 'hits': [{'_index': 'shakespeare', '_id': '0', '_score': 1.0, '_source': {'type': 'act', 'line_id': 1, 'play_name': 'Henry IV', 'speech_number': '', 'line_number': '', 'speaker': '', 'text_entry': 'ACT I'}}, {'_index': 'shakespeare', '_id': '1', '_score': 1.0, '_source': {'type': 'scene', 'line_id': 2, 'play_name': 'Henry IV', 'speech_number': '', 'line_number': '', 'speaker': '', 'text_entry': 'SCENE I. London. The palace.'}}, {'_index': 'shakespeare', '_id': '2', '_score': 1.0, '_source': {'type': 'line', 'line_id': 3, 'play_name': 'Henry IV', 'speech_number': '', 'line_number': '', 'speaker': '', 'text_entry': 'Enter KING HENRY, LORD JOHN OF LANCASTER, the EARL of WESTMORELAND, SIR WALTER BLUNT, and others'}}, {'_index': 'shakespeare', '_id': '3', '_score': 1.0, '_source': {'type': 'line', 'line_id': 4, 'play_name': 'Henry IV', 'speech_number': 1, 'line_number': '1.1.1', 'speaker': 'KING HENRY IV', 'text_entry': 'So sh

### Búsqueda por campos

También podemos hacer búsquedas directas en cada campo:

In [11]:
search_query("love AND speaker:(ROMEO OR JULIET)")

{'total': {'value': 75, 'relation': 'eq'}, 'max_score': 10.849649, 'hits': [{'_index': 'shakespeare', '_id': '86302', '_score': 10.849649, '_source': {'type': 'line', 'line_id': 86303, 'play_name': 'Romeo and Juliet', 'speech_number': 41, 'line_number': '2.2.165', 'speaker': 'ROMEO', 'text_entry': 'Love goes toward love, as schoolboys from'}}, {'_index': 'shakespeare', '_id': '86205', '_score': 10.439217, '_source': {'type': 'line', 'line_id': 86206, 'play_name': 'Romeo and Juliet', 'speech_number': 13, 'line_number': '2.2.72', 'speaker': 'ROMEO', 'text_entry': 'And what love can do that dares love attempt;'}}, {'_index': 'shakespeare', '_id': '86304', '_score': 10.439217, '_source': {'type': 'line', 'line_id': 86305, 'play_name': 'Romeo and Juliet', 'speech_number': 41, 'line_number': '2.2.167', 'speaker': 'ROMEO', 'text_entry': 'But love from love, toward school with heavy looks.'}}, {'_index': 'shakespeare', '_id': '86436', '_score': 10.439217, '_source': {'type': 'line', 'line_id':

En este caso 👆 hemos utilizado el operador **AND** para que busque frases en las que aparezca *love* además de el operador **OR** referido al campo *speaker* para que quien lo dice sea o ***Romeo o Julieta*** 😍

In [12]:
search_query("speaker:(ROMEO OR JULIET) AND NOT speaker:JULIET")

{'total': {'value': 651, 'relation': 'eq'}, 'max_score': 5.1415777, 'hits': [{'_index': 'shakespeare', '_id': '85464', '_score': 5.1415777, '_source': {'type': 'line', 'line_id': 85465, 'play_name': 'Romeo and Juliet', 'speech_number': 64, 'line_number': '1.1.153', 'speaker': 'ROMEO', 'text_entry': 'Is the day so young?'}}, {'_index': 'shakespeare', '_id': '85466', '_score': 5.1415777, '_source': {'type': 'line', 'line_id': 85467, 'play_name': 'Romeo and Juliet', 'speech_number': 66, 'line_number': '1.1.155', 'speaker': 'ROMEO', 'text_entry': 'Ay me! sad hours seem long.'}}, {'_index': 'shakespeare', '_id': '85467', '_score': 5.1415777, '_source': {'type': 'line', 'line_id': 85468, 'play_name': 'Romeo and Juliet', 'speech_number': 66, 'line_number': '1.1.156', 'speaker': 'ROMEO', 'text_entry': 'Was that my father that went hence so fast?'}}, {'_index': 'shakespeare', '_id': '85469', '_score': 5.1415777, '_source': {'type': 'line', 'line_id': 85470, 'play_name': 'Romeo and Juliet', 'spe

En este caso 👆 le decimos que nos muestre sentencias donde los oradores sean Romeo O Julieta Y que no sea Julieta 😠.
(Sería lo mismo que decir que sea solo Romeo, pero así vemos cómo funciona)

## Búsqueda con boosts

También podemos hacer búsquedas con ciertos campos potenciados.

Podemos asignar prioridad o importancia (pesos) a las búsquedas.
En Elasticsearch, el concepto de "**boost**" se refiere a la capacidad de **aumentar la relevancia de ciertos documentos o términos dentro de una consult**a.

Estos boosts se pueden aplicar a los campos, a los términos y a las queries completas. Veamos un boost de campo:

In [13]:
search_query("text_entry:love AND (speaker:ROMEO^4 OR speaker:JULIET^5)")

{'total': {'value': 75, 'relation': 'eq'}, 'max_score': 30.645538, 'hits': [{'_index': 'shakespeare', '_id': '86325', '_score': 30.645538, '_source': {'type': 'line', 'line_id': 86326, 'play_name': 'Romeo and Juliet', 'speech_number': 50, 'line_number': '2.2.186', 'speaker': 'JULIET', 'text_entry': 'Remembering how I love thy company.'}}, {'_index': 'shakespeare', '_id': '87028', '_score': 30.645538, '_source': {'type': 'line', 'line_id': 87029, 'play_name': 'Romeo and Juliet', 'speech_number': 1, 'line_number': '3.2.16', 'speaker': 'JULIET', 'text_entry': 'Think true love acted simple modesty.'}}, {'_index': 'shakespeare', '_id': '59380', '_score': 30.394375, '_source': {'type': 'line', 'line_id': 59381, 'play_name': 'Measure for measure', 'speech_number': 20, 'line_number': '2.3.44', 'speaker': 'JULIET', 'text_entry': 'Must die to-morrow! O injurious love,'}}, {'_index': 'shakespeare', '_id': '86055', '_score': 30.394375, '_source': {'type': 'line', 'line_id': 86056, 'play_name': 'Ro

En esta query 👆 le hemos dicho que Romeo tenga un peso 4 veces el peso normal y Juliet, 5 veces el peso normal, por lo que ha priorizado Juliet, que ya de por sí, seguro que aparece muchas veces.

### Búsqueda con comodines o wildcards

Podemos hacer búsquedas con comodines dentro de campos:

In [14]:
search_query("play_name:Ot?ello")

{'total': {'value': 3762, 'relation': 'eq'}, 'max_score': 1.0, 'hits': [{'_index': 'shakespeare', '_id': '72000', '_score': 1.0, '_source': {'type': 'act', 'line_id': 72001, 'play_name': 'Othello', 'speech_number': 59, 'line_number': '', 'speaker': 'BENEDICK', 'text_entry': 'ACT I'}}, {'_index': 'shakespeare', '_id': '72001', '_score': 1.0, '_source': {'type': 'scene', 'line_id': 72002, 'play_name': 'Othello', 'speech_number': 59, 'line_number': '', 'speaker': 'BENEDICK', 'text_entry': 'SCENE I. Venice. A street.'}}, {'_index': 'shakespeare', '_id': '72002', '_score': 1.0, '_source': {'type': 'line', 'line_id': 72003, 'play_name': 'Othello', 'speech_number': 59, 'line_number': '', 'speaker': 'BENEDICK', 'text_entry': 'Enter RODERIGO and IAGO'}}, {'_index': 'shakespeare', '_id': '72003', '_score': 1.0, '_source': {'type': 'line', 'line_id': 72004, 'play_name': 'Othello', 'speech_number': 1, 'line_number': '1.1.1', 'speaker': 'RODERIGO', 'text_entry': 'Tush! never tell me; I take it much

In [16]:
search_query("play_name:K* AND NOT play_name:*Lear")

{'total': {'value': 2766, 'relation': 'eq'}, 'max_score': 1.0, 'hits': [{'_index': 'shakespeare', '_id': '43490', '_score': 1.0, '_source': {'type': 'act', 'line_id': 43491, 'play_name': 'King John', 'speech_number': 11, 'line_number': '5.5.97', 'speaker': 'KING HENRY VIII', 'text_entry': 'ACT I'}}, {'_index': 'shakespeare', '_id': '43491', '_score': 1.0, '_source': {'type': 'scene', 'line_id': 43492, 'play_name': 'King John', 'speech_number': 11, 'line_number': '5.5.97', 'speaker': 'KING HENRY VIII', 'text_entry': 'SCENE I. KING JOHNS palace.'}}, {'_index': 'shakespeare', '_id': '43492', '_score': 1.0, '_source': {'type': 'line', 'line_id': 43493, 'play_name': 'King John', 'speech_number': 11, 'line_number': '', 'speaker': 'KING HENRY VIII', 'text_entry': 'Enter KING JOHN, QUEEN ELINOR, PEMBROKE, ESSEX, SALISBURY, and others, with CHATILLON'}}, {'_index': 'shakespeare', '_id': '43493', '_score': 1.0, '_source': {'type': 'line', 'line_id': 43494, 'play_name': 'King John', 'speech_numbe

El carácter "**?**" actúa como comodín de un solo carácter, mientras que "*" indica múltiples caracteres.

En el primer caso, vemos que la "**?**" está en el lugar donde debía ir la "h" de Othelo.

En el segundo, lo que estamos haciendo es buscar documentos donde el campo play_name comience con la letra "K" y que no contenga la palabra "Lear" en ninguna posición.

### Búsqueda con Fuzziness

La opción de **fuzziness** en Elasticsearch se utiliza para **buscar términos que son similares a un término dado**, incluso si contienen errores de ortografía o están ligeramente modificados. Es útil cuando se desea recuperar resultados que pueden tener pequeñas variaciones respecto al término buscado originalmente.

Permite ajustar cómo se manejan las discrepancias entre los términos de búsqueda y los términos indexados en el índice. Funciona configurando un parámetro numérico que indica el grado de flexibilidad permitido en la búsqueda de términos similares.

Los valores que se le pueden asignar son 0, 1 y 2:



*   0: **No se permiten errores.** Buscará exactamente el término especificado en el índice. No habrá correcciones o tolerancia para errores de ortografía o variaciones.
*   1: **Se perimte un error único**. Buscará términos que estén a una distancia de edición de un carácter del término especificado. Esto incluye cambios simples como sustituciones, inserciones o eliminaciones de un solo carácter.
*   2: Permite dos errores: Buscará términos que estén a una distancia de edición de hasta dos caracteres del término especificado. Esto puede incluir cambios más significativos en la ortografía o la estructura del término.



Por ejemplo:

In [21]:
search_query("play_name:Othello~0")

{'total': {'value': 3762, 'relation': 'eq'}, 'max_score': 3.3880167, 'hits': [{'_index': 'shakespeare', '_id': '72000', '_score': 3.3880167, '_source': {'type': 'act', 'line_id': 72001, 'play_name': 'Othello', 'speech_number': 59, 'line_number': '', 'speaker': 'BENEDICK', 'text_entry': 'ACT I'}}, {'_index': 'shakespeare', '_id': '72001', '_score': 3.3880167, '_source': {'type': 'scene', 'line_id': 72002, 'play_name': 'Othello', 'speech_number': 59, 'line_number': '', 'speaker': 'BENEDICK', 'text_entry': 'SCENE I. Venice. A street.'}}, {'_index': 'shakespeare', '_id': '72002', '_score': 3.3880167, '_source': {'type': 'line', 'line_id': 72003, 'play_name': 'Othello', 'speech_number': 59, 'line_number': '', 'speaker': 'BENEDICK', 'text_entry': 'Enter RODERIGO and IAGO'}}, {'_index': 'shakespeare', '_id': '72003', '_score': 3.3880167, '_source': {'type': 'line', 'line_id': 72004, 'play_name': 'Othello', 'speech_number': 1, 'line_number': '1.1.1', 'speaker': 'RODERIGO', 'text_entry': 'Tush!

In [22]:
search_query("play_name:Ophello~0")

{'total': {'value': 0, 'relation': 'eq'}, 'max_score': None, 'hits': []}
Total 0 found in 1ms
--------------------------------------------------------------------------------


In [28]:
search_query("play_name:Ophello~1")

{'total': {'value': 3762, 'relation': 'eq'}, 'max_score': 2.904014, 'hits': [{'_index': 'shakespeare', '_id': '72000', '_score': 2.904014, '_source': {'type': 'act', 'line_id': 72001, 'play_name': 'Othello', 'speech_number': 59, 'line_number': '', 'speaker': 'BENEDICK', 'text_entry': 'ACT I'}}, {'_index': 'shakespeare', '_id': '72001', '_score': 2.904014, '_source': {'type': 'scene', 'line_id': 72002, 'play_name': 'Othello', 'speech_number': 59, 'line_number': '', 'speaker': 'BENEDICK', 'text_entry': 'SCENE I. Venice. A street.'}}, {'_index': 'shakespeare', '_id': '72002', '_score': 2.904014, '_source': {'type': 'line', 'line_id': 72003, 'play_name': 'Othello', 'speech_number': 59, 'line_number': '', 'speaker': 'BENEDICK', 'text_entry': 'Enter RODERIGO and IAGO'}}, {'_index': 'shakespeare', '_id': '72003', '_score': 2.904014, '_source': {'type': 'line', 'line_id': 72004, 'play_name': 'Othello', 'speech_number': 1, 'line_number': '1.1.1', 'speaker': 'RODERIGO', 'text_entry': 'Tush! neve

In [30]:
search_query("play_name:Opello~2")

{'total': {'value': 3762, 'relation': 'eq'}, 'max_score': 2.2586775, 'hits': [{'_index': 'shakespeare', '_id': '72000', '_score': 2.2586775, '_source': {'type': 'act', 'line_id': 72001, 'play_name': 'Othello', 'speech_number': 59, 'line_number': '', 'speaker': 'BENEDICK', 'text_entry': 'ACT I'}}, {'_index': 'shakespeare', '_id': '72001', '_score': 2.2586775, '_source': {'type': 'scene', 'line_id': 72002, 'play_name': 'Othello', 'speech_number': 59, 'line_number': '', 'speaker': 'BENEDICK', 'text_entry': 'SCENE I. Venice. A street.'}}, {'_index': 'shakespeare', '_id': '72002', '_score': 2.2586775, '_source': {'type': 'line', 'line_id': 72003, 'play_name': 'Othello', 'speech_number': 59, 'line_number': '', 'speaker': 'BENEDICK', 'text_entry': 'Enter RODERIGO and IAGO'}}, {'_index': 'shakespeare', '_id': '72003', '_score': 2.2586775, '_source': {'type': 'line', 'line_id': 72004, 'play_name': 'Othello', 'speech_number': 1, 'line_number': '1.1.1', 'speaker': 'RODERIGO', 'text_entry': 'Tush!

In [37]:
search_query("play_name:Kiong~0 John~0")

{'total': {'value': 222, 'relation': 'eq'}, 'max_score': 9.806491, 'hits': [{'_index': 'shakespeare', '_id': '64627', '_score': 9.806491, '_source': {'type': 'line', 'line_id': 64628, 'play_name': 'Merry Wives of Windsor', 'speech_number': 21, 'line_number': '1.4.50', 'speaker': 'MISTRESS QUICKLY', 'text_entry': 'What, John Rugby! John!'}}, {'_index': 'shakespeare', '_id': '65709', '_score': 9.806491, '_source': {'type': 'line', 'line_id': 65710, 'play_name': 'Merry Wives of Windsor', 'speech_number': 52, 'line_number': '3.3.123', 'speaker': 'MISTRESS FORD', 'text_entry': 'What, John! Robert! John!'}}, {'_index': 'shakespeare', '_id': '64609', '_score': 9.589895, '_source': {'type': 'line', 'line_id': 64610, 'play_name': 'Merry Wives of Windsor', 'speech_number': 15, 'line_number': '1.4.35', 'speaker': 'MISTRESS QUICKLY', 'text_entry': 'What, John Rugby! John! what, John, I say!'}}, {'_index': 'shakespeare', '_id': '2039', '_score': 8.2047, '_source': {'type': 'line', 'line_id': 2040, 

### Indexes y maps


**Vamos a crear un nuevo índice y su mapeo.**

Antes de insertar datos es recomendable siempre crear el indice y sus mapeos:

In [38]:
es.indices.create(index='documents_october', body={
  "settings": {
    "number_of_replicas": 1,
    "number_of_shards": 3,
    "refresh_interval": "1s"
  },
  "mappings": {
    "properties": {
      "title": {
        "type": "text",
        "analyzer": "english"
      }
    }
  }
})

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'documents_october'})

Esto también lo podríamos hacer con la librería **request y los métodos PUT y GET** como hicimos en el anterior notebook (aunque para algo tenemos la librería elasticsearch que es específica para ello).

Pero por ver un ejemplo, este mismo índice deberíamos crearlo así:

```
server_ip = '<tu_direccion_ip>'
auth = ('usuario', 'contraseña')
index_name = 'documents_march'

# URL para la creación del índice
url = f'https://{server_ip}:9200/{index_name}'

# Configuración y mapeo del índice
settings = {
    "settings": {
        "number_of_replicas": 1,
        "number_of_shards": 3,
        "refresh_interval": "1s"
    },
    "mappings": {
        "properties": {
            "title": {
                "type": "text",
                "analyzer": "english"
            }
        }
    }
}

# Realiza la solicitud PUT a Elasticsearch
response = requests.put(url, auth=auth, json=settings, verify=False)

```

Como se puede ver, con la librería elasticsearch, es más sencillo. aunque la esencia es la misma.

In [39]:
es.indices.put_mapping(index='documents_october', body={
    "properties": {
      "tag": {
          "type": "keyword"
      },
    }
})

ObjectApiResponse({'acknowledged': True})

In [40]:
es.indices.get(index="documents_october")

ObjectApiResponse({'documents_october': {'aliases': {}, 'mappings': {'properties': {'tag': {'type': 'keyword'}, 'title': {'type': 'text', 'analyzer': 'english'}}}, 'settings': {'index': {'routing': {'allocation': {'include': {'_tier_preference': 'data_content'}}}, 'refresh_interval': '1s', 'number_of_shards': '3', 'provided_name': 'documents_october', 'creation_date': '1729715477111', 'number_of_replicas': '1', 'uuid': '2WVDpl0RS2eLyeYf6IxadQ', 'version': {'created': '8505000'}}}}})

In [41]:
# Le agregamos un documento al índice (para que no esté vacío)

# Indexar un documento en el índice 'documents_october'
response = es.index(index='documents_october', body={
    "title": "Documento de Ejemplo",
    "content": "Este es un ejemplo de contenido",
    "tag": ["ejemplo", "documento"]
})
print("Documento indexado con ID:", response['_id'])

Documento indexado con ID: dT0Xu5IB_TZJqeSlwVcO


Esto con requests sería equivalente a

```
response = requests.get(url, auth=auth, verify=False)

```


## Manejo de documentos

Vamos a hacer **CRUD** sobre documentos.

¿CRUD?!!!!

*   CREATE
*   READ
*   UPDATE
*   DELETE

En las operaciones CRUD, será necesario indicar siempre el index y el id, para poder identificar de manera única el documento dentro del índice. Para que quede más claro:

**Índice**

El índice en Elasticsearch es análogo a una base de datos en un sistema de gestión de bases de datos relacional. Cada índice puede contener múltiples documentos y cada documento debe pertenecer a un índice específico. Indicar el índice le dice a Elasticsearch dónde buscar el documento.

**ID**

El ID es un identificador único para un documento dentro de un índice. Aunque varios documentos en diferentes índices pueden tener el mismo ID, cada documento dentro de un índice específico debe tener un ID único. Esto permite a Elasticsearch identificar y operar en un documento específico dentro de ese índice.

Un ejemplo ->
Supongamos que tenemos varios índices para diferentes tipos de datos, como alumnos, profesores y cursos. Si queremos realizar una operación sobre un documento específico de alumnos, tenemos que especificar el índice alumnos para que Elasticsearch sepa dónde buscar. Además, hay que proporcionar el ID del documento para que Elasticsearch sepa cuál de los muchos documentos dentro del índice alumnos queremos operar.

Si no indico ambos -> Elasticsearch no sabría en qué conjunto de datos (índice) buscar el documento o en qué documento específico dentro del índice devolver.

Cuando agregamos un documento a un index, como hicimos arriba, si no le indico un id, se lo asignará de forma automática.

### Creación:

In [42]:
es.create(index='alumnos', id=6, body={
    "title": "New Document",
    "content": "This is a new document for the master class",
    "tag": ["general", "testing"]
})

ObjectApiResponse({'_index': 'alumnos', '_id': '6', '_version': 1, 'result': 'created', '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 0, '_primary_term': 1})

### Lectura

In [43]:
es.get(index='alumnos', id=6)

ObjectApiResponse({'_index': 'alumnos', '_id': '6', '_version': 1, '_seq_no': 0, '_primary_term': 1, 'found': True, '_source': {'title': 'New Document', 'content': 'This is a new document for the master class', 'tag': ['general', 'testing']}})

### Borrado

Si no sabemos el id del index que queremos borrar:

In [44]:
# Realizar una búsqueda en el índice
response = es.search(index='documents_october')

# Imprimir los IDs de los documentos encontrados
for hit in response['hits']['hits']:
    doc_id = hit['_id']
    print(f"Found document with ID: {doc_id}")

Found document with ID: dT0Xu5IB_TZJqeSlwVcO


In [45]:
es.delete(index='documents_october', id='dT0Xu5IB_TZJqeSlwVcO')

ObjectApiResponse({'_index': 'documents_october', '_id': 'dT0Xu5IB_TZJqeSlwVcO', '_version': 2, 'result': 'deleted', '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 1, '_primary_term': 1})


Ya hemos visto todas las operaciones base en **ElasticSearch**. A partir de aquí, igual que en SQL, todo dependerá de los resultados que queramos o de la tarea que debamos resolver, que harán que las consultas se vayan haciendo más complejas.

Como todo, a partir de aquí, es cuestión de práctica!

# **Vemos Kibana????**

https://www.elastic.co/es/demos

Entremos a nuestro Kibana 🌈🦄:

http://IP_SERVER_ELASTIC:5601